# Attention-MADE on a Colab GPU

Connect this notebook to a **Colab GPU** kernel (Select Kernel → Colab → pick GPU, not CPU), then run the cells in order.

Mount Drive **before** training. The train command then copies the best validation checkpoint to `MyDrive/made-attention/best.ckpt` whenever `/content/drive/MyDrive` exists. The epoch-named file is saved next to it; `best_checkpoint.json` records which `epoch-XXXX.ckpt` that is.

In [1]:
!uv pip install -q torch lightning cyclopts torchmetrics

In [2]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. In Cursor: Select Kernel → Colab → choose a GPU runtime, then rerun."
)
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__)

Tesla T4
torch 2.11.0+cu128


In [4]:
from pathlib import Path

REPO = "https://github.com/ml-and-ds-degree/deep-generative-models-of-texts-and-images.git"
BRANCH = "made-attention"
ROOT = Path("/content/deep-generative-models-of-texts-and-images")

if not (ROOT / "src" / "made_reproduction").exists():
    !git clone --branch {BRANCH} --depth 1 "{REPO}" "{ROOT}"

%cd {ROOT}
!git rev-parse --abbrev-ref HEAD && git log -1 --oneline

Cloning into '/content/deep-generative-models-of-texts-and-images'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 78 (delta 1), reused 41 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 2.39 MiB | 31.02 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/deep-generative-models-of-texts-and-images
made-attention
1ca1b72 (grafted, HEAD -> made-attention, origin/made-attention) feat(made): add residual attention MADE and a Colab GPU notebook


In [5]:
from pathlib import Path
import os


os.environ["PYTHONPATH"] = str(Path.cwd() / "src")
print("PYTHONPATH", os.environ["PYTHONPATH"])

PYTHONPATH /content/deep-generative-models-of-texts-and-images/src


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive ready; training will export to MyDrive/made-attention/best.ckpt")

In [ ]:
!PYTHONPATH=src python -m made_reproduction.cli train binarized-mnist \
  --architecture attention \
  --accelerator gpu \
  --max-epochs 25 \
  --num-workers 2

Seed set to 1234
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model    │ AttentionMADE │  1.8 M │ train │     0 │
│ 1 │ test_nll │ MeanMetric    │      0 │ train │     0 │
└───┴──────────┴───────────────┴────────┴───────┴───────┘
Trainable params: 1.8 M                                                         
Non-trainable params: 0                                                         
Total params: 1.8 M                                                             
Total estimated model params size (MB): 

In [ ]:
from pathlib import Path

ckpt_dir = Path("outputs/made/binarized_mnist_attention/checkpoints")
ckpts = sorted(ckpt_dir.glob("epoch-*.ckpt"))
assert ckpts, "No epoch checkpoint yet; wait for training to finish."
best = ckpts[-1]
print("evaluating", best)

!PYTHONPATH=src python -m made_reproduction.cli evaluate "{best}" binarized-mnist --accelerator gpu

In [ ]:
# Fallback only: skip this if training already printed
# "exported best checkpoint ... -> .../best.ckpt"
from pathlib import Path
import shutil

ckpt_dir = Path("outputs/made/binarized_mnist_attention/checkpoints")
best_ckpts = sorted(ckpt_dir.glob("epoch-*.ckpt"))
assert best_ckpts, f"No best checkpoint in {ckpt_dir.resolve()}"
best = best_ckpts[-1]
dest = Path("/content/drive/MyDrive/made-attention")
dest.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, dest / "best.ckpt")
shutil.copy2(best, dest / best.name)
print("copied", best.name, "and best.ckpt ->", dest)